<a href="https://colab.research.google.com/github/Maurofome/Lab01_EDA_Aliaga_Mauricio/blob/main/Lab2_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fase 1: Carga

Usé el dataset Groceries dataset for Market Basket Analysis by Rashik Rahman

Tiene ~15.000 transacciones, cumple el rango pedido y se explica en una frase: “compras de supermercado de distintos clientes”.

link: https://www.kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba

In [3]:
import kagglehub, pandas as pd, os

path = kagglehub.dataset_download("rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba")
print(os.listdir(path))

100%|██████████| 455k/455k [00:00<00:00, 68.8MB/s]

Extracting files...
['Groceries data.csv', 'basket.csv']


In [4]:
archivo = os.listdir(path)[0]   # si hay más de un archivo, escribe el nombre exacto
df = pd.read_csv(os.path.join(path, archivo))
print(df.shape)
df.head()

(38765, 7)


,Member_number,Date,itemDescription,year,month,day,day_of_week
0,1808,2015-07-21,tropical fruit,2015,7,21,1
1,2552,2015-05-01,whole milk,2015,5,1,4
2,2300,2015-09-19,pip fruit,2015,9,19,5
3,1187,2015-12-12,other vegetables,2015,12,12,5
4,3037,2015-01-02,whole milk,2015,1,2,4


Transacción = Una compra de un cliente.
Ítem = Un producto del supermercado.


Para este laboratorio usé el dataset Groceries dataset for Market Basket Analysis (MBA) de Kaggle, que lo descargué con kagglehub usando Colab Secrets.

Transacción: una compra de un cliente en el supermercado.
Ítem: un producto que el cliente compró.
Contexto: compras de un supermercado, donde cada compra tiene uno o más productos.

El dataset tiene [N] transacciones y [N] ítems distintos.

Fase 2: Preparación

Paso 2.1: Armar las canastas. Este código sirve para los dos formatos posibles:

In [5]:
if 'itemDescription' in df.columns:
    # Formato largo: una fila por producto comprado
    df['itemDescription'] = df['itemDescription'].astype(str).str.strip().str.lower()
    df['transaccion'] = df['Member_number'].astype(str) + "_" + df['Date'].astype(str)
    print("Nulos:\n", df.isnull().sum())
    print("Duplicados:", df.duplicated(['transaccion', 'itemDescription']).sum())
    df = df.drop_duplicates(['transaccion', 'itemDescription'])
    canastas = df.groupby('transaccion')['itemDescription'].apply(list).tolist()
else:
    # Formato ancho: una fila por compra, productos en columnas
    print("Nulos por columna:\n", df.isnull().sum())
    canastas = (df.apply(lambda fila: sorted({str(x).strip().lower()
                         for x in fila if pd.notna(x) and str(x).strip() != ""}), axis=1)
                  .tolist())
    canastas = [c for c in canastas if len(c) > 0]   # saca filas vacías

print("Transacciones:", len(canastas))
print("Ítems distintos:", len({i for c in canastas for i in c}))

Nulos:
 Member_number      0
Date               0
itemDescription    0
year               0
month              0
day                0
day_of_week        0
transaccion        0
dtype: int64
Duplicados: 759
Transacciones: 14963
Ítems distintos: 167


Paso 2.2: Revisar nombres inconsistentes

In [6]:
from collections import Counter
conteo = Counter(i for c in canastas for i in c)
print(sorted(conteo.items(), key=lambda x: x[1])[:15])   # los más raros

[('preservation products', 1), ('kitchen utensil', 1), ('baby cosmetics', 3), ('bags', 4), ('make up remover', 5), ('frozen chicken', 5), ('toilet cleaner', 5), ('rubbing alcohol', 5), ('salad dressing', 6), ('whisky', 8), ('hair spray', 9), ('liqueur', 9), ('decalcifier', 9), ('organic products', 10), ('frozen fruits', 11)]


Paso 2.3: One-hot encoding

In [7]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
matriz = te.fit(canastas).transform(canastas)
basket = pd.DataFrame(matriz, columns=te.columns_)
basket.head()

,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,...,uht-milk,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Apriori no lee listas de productos. Necesita saber, para cada compra y cada producto, si estaba (True) o no (False). Con esa tabla cuenta rápido cuántas compras contienen cada combinación.

Revisé los datos antes de usarlos:

Nulos: [no encontré / encontré N y los eliminé].
Nombres inconsistentes: [no encontré / corregí N nombres, pasándolos a minúscula y quitando espacios].
Duplicados: [eliminé N productos repetidos dentro de una misma compra].

Después transformé los datos a una tabla con una fila por transacción y una columna por producto, con valores verdadero/falso (one-hot encoding).

¿Por qué Apriori necesita este formato? Porque no puede leer listas de productos. Necesita saber, para cada compra, si un producto estaba (True) o no (False). Con esa tabla cuenta rápido cuántas compras contienen cada combinación.

Fase 3: Itemsets frecuentes

In [8]:
frecuencia = basket.mean().sort_values(ascending=False)
print(frecuencia.head(10))
print(frecuencia.describe())

whole milk          0.157923
other vegetables    0.122101
rolls/buns          0.110005
soda                0.097106
yogurt              0.085879
root vegetables     0.069572
tropical fruit      0.067767
bottled water       0.060683
sausage             0.060349
citrus fruit        0.053131
dtype: float64
count    167.000000
mean       0.015210
std        0.023381
min        0.000067
25%        0.002038
50%        0.005681
75%        0.017644
max        0.157923
dtype: float64


In [9]:
from mlxtend.frequent_patterns import apriori, association_rules

itemsets = apriori(basket, min_support=0.01, use_colnames=True)
print(len(itemsets))
itemsets.sort_values('support', ascending=False).head(10)

69


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets
62,0.157923,(whole milk)
39,0.122101,(other vegetables)
45,0.110005,(rolls/buns)
51,0.097106,(soda)
63,0.085879,(yogurt)
46,0.069572,(root vegetables)
56,0.067767,(tropical fruit)
4,0.060683,(bottled water)
48,0.060349,(sausage)
14,0.053131,(citrus fruit)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Antes de elegir el soporte mínimo miré qué tan frecuentes son los productos. Los más comprados aparecen en cerca del [XX]% de las compras, y la mayoría aparece en menos del [X]%.

Por eso elegí un soporte mínimo de [0.01]: un valor alto dejaría casi sin resultados, y uno muy bajo generaría demasiados itemsets y podría agotar la memoria de Colab.

Con ese valor obtuve [N] itemsets frecuentes. Los más frecuentes son [producto 1], [producto 2] y [producto 3].

Fase 4: Reglas y experimento obligatorio

In [10]:
reglas = association_rules(itemsets, metric="confidence", min_threshold=0.10)
reglas = reglas.sort_values('lift', ascending=False)
reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
3,(yogurt),(whole milk),0.011161,0.129961,0.822940
1,(rolls/buns),(whole milk),0.013968,0.126974,0.804028
0,(other vegetables),(whole milk),0.014837,0.121511,0.769430
2,(soda),(whole milk),0.011629,0.119752,0.758296


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Comparación de 3 configuraciones:

In [ ]:
configs = [(0.01, 0.10), (0.005, 0.10), (0.003, 0.05)]
filas = []
for s, c in configs:
    its = apriori(basket, min_support=s, use_colnames=True)
    if (its['itemsets'].apply(len) >= 2).any():
        r = association_rules(its, metric="confidence", min_threshold=c)
    else:
        r = pd.DataFrame()
    filas.append({"soporte_min": s, "confianza_min": c,
                  "n_itemsets": len(its), "n_reglas": len(r)})

comparacion = pd.DataFrame(filas)
comparacion

Generé las reglas con una confianza mínima de [0.10] y las ordené por lift, porque muestra si dos productos se compran juntos más de lo normal (lift mayor a 1), sin depender de que un producto sea popular.

Conclusión: al bajar el soporte aparecen más reglas, pero de productos poco comprados y menos confiables. Al subirlo hay pocas reglas, pero con más respaldo. Elegí la configuración [X] porque da un buen número de reglas con buen respaldo.

Fase 5: Interpretación

In [11]:
reglas[reglas['lift'] < 1.05][['antecedents', 'consequents', 'support', 'confidence', 'lift']].head()

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,support,confidence,lift
3,(yogurt),(whole milk),0.011161,0.129961,0.822940
1,(rolls/buns),(whole milk),0.013968,0.126974,0.804028
0,(other vegetables),(whole milk),0.014837,0.121511,0.769430
2,(soda),(whole milk),0.011629,0.119752,0.758296


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


5. Interpretación
Conclusión de los umbrales

Probé 3 configuraciones. Al bajar el soporte mínimo salieron más reglas, pero de productos poco comprados. Al subirlo salieron pocas, pero más seguras. Usé [la configuración elegida] porque da un buen número de reglas con buen respaldo.

5 reglas más valiosas
[A] → [B] (lift [ ]): quien compra [A] suele llevar también [B].
[C] → [D] (lift [ ]): quien compra [C] suele llevar también [D].
[E] → [F] (lift [ ]): quien compra [E] suele llevar también [F].
[G] → [H] (lift [ ]): quien compra [G] suele llevar también [H].
[I] → [J] (lift [ ]): quien compra [I] suele llevar también [J].
Regla que descartaría

Descartaría [X] → [Y]. Su confianza es [ ], pero su lift es [≈1], o sea que comprar [X] no cambia la probabilidad de comprar [Y]. Pasa porque [Y] es un producto que compra mucha gente.

Decisión para el administrador

Poner [A] y [B] juntos en el local y hacer una promoción con [C] y [D].

Reflexión final

Aprendí que los umbrales cambian mucho los resultados. También que la confianza sola puede engañar y que el lift ayuda a ver si la relación es real. Verifiqué una regla a mano y coincidió con la librería. Las reglas muestran qué se compra junto, no predicen nada.